# Lenguajes y Autómatas II (SCD-1016)
## Sesión 30 · Hito 3: Generador de TAC y Optimizador End-to-End

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Qué tan pequeño y correcto puede quedar el programa intermedio antes de convertirlo en ensamblador?
>
> **Objetivo de la sesión:** Integrar la traducción de un AST a TAC con el plegado de constantes, devolviendo el lugar donde queda el resultado y las instrucciones que sí hacen falta.

**Entregable:** tu función generar_tac_opt() en artesano_sesion30_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion30.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes un árbol con hojas numéricas, variables y operadores binarios. Debes emitir solo las instrucciones necesarias: si un subárbol es constante, se calcula en compilación y no genera línea. Además, escribe en resultado el literal o el temporal que contiene el valor final.

Al final de la sesión tendrás: **Una función en C que traduce un AST a TAC optimizado, pliega constantes y reporta el lugar del resultado.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 30](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion30.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion30.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion30.h"

int generar_tac_opt(const Nodo *raiz, char lineas[][64], int max, char *resultado, int cap);
```

- Un subárbol con dos hojas numéricas se pliega a un literal y no emite instrucciones.
- Las identidades x + 0, x * 1 y x * 0 se resuelven sin generar una operación.
- No pliegues una división entre cero: deja que la instrucción llegue al backend.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion30_c.c` junto con `artesano_sesion30_c.c`
> (o `ia_sesion30_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `x + 2 * 3 (linea)` | `_t0 = x + 6` |
| 2 | `x + 2 * 3 (resultado)` | `_t0` |
| 3 | `x * y + 2 * 3` | `_t1 = _t0 + 6` |
| 4 | `todo constante: sin lineas` | 0 |
| 5 | `todo constante: resultado` | `45` |
| 6 | `x + 0 (identidad)` | `x` |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 30 - AST a TAC optimizado de punta a punta ===
Caso                               esperado:              obtenido:              resultado
[ 1] x + 2 * 3 (linea)                  esperado: _t0 = x + 6  obtenido: _t0 = x + 6  OK
[ 2] x + 2 * 3 (resultado)              esperado: _t0          obtenido: _t0          OK
[ 3] x * y + 2 * 3
     esperado: _t1 = _t0 + 6
     obtenido: _t1 = _t0 + 6
     -> OK
[ 4] todo constante: sin lineas         esperado: 0            obtenido: 0            OK
[ 5] todo constante: resultado          esperado: 45           obtenido: 45           OK
[ 6] x + 0 (identidad)                  esperado: x            obtenido: x            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion30.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Un subárbol constante se sustituye por su valor y no genera instrucciones.
- [ ] El lugar del resultado coincide con el último temporal emitido o con el literal plegado.
- [ ] El TAC optimizado conserva exactamente el valor que produciría el árbol original.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion30_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion30.h
/* api_sesion30.h - contrato de la Sesion 30
 * AST a TAC optimizado de punta a punta
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION30_H
#define API_SESION30_H

typedef struct Nodo {
    char tipo;
    char op;
    char valor[16];
    struct Nodo *izq;
    struct Nodo *der;
} Nodo;

int generar_tac_opt(const Nodo *raiz, char lineas[][64], int max, char *resultado, int cap);

#endif /* API_SESION30_H */


In [ ]:
%%writefile artesano_sesion30_c.c
/* artesano_sesion30_c.c - Fase 1: El Artesano (sin IA)
 * AST a TAC optimizado de punta a punta.
 *
 * Implementa lo que declara api_sesion30.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion30.h"
#include <stdio.h>
#include <string.h>

int generar_tac_opt(const Nodo *raiz, char lineas[][64], int max, char *resultado, int cap) {
          (void)raiz; (void)lineas; (void)max; (void)resultado; (void)cap; /* TODO: traducir y plegar */
          return 0;
      }


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion30_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion30_c.c
/* ia_sesion30_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion30.h. No puede traer main().
 */
#include "api_sesion30.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion30_c.c
/* harness_sesion30_c.c - auditoria automatica de la Sesion 30
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion30_c.c artesano_sesion30_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion30.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_str_largo(const char *caso, const char *esperado,
                                const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %s\n", probadas, caso);
    printf("     esperado: %s\n", esperado);
    printf("     obtenido: %s\n", obtenido ? obtenido : "(NULL)");
    printf("     -> %s\n", ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 30 - AST a TAC optimizado de punta a punta ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { Nodo x = {.tipo='V', .valor="x"}; Nodo n2 = {.tipo='N', .valor="2"}; Nodo n3 = {.tipo='N', .valor="3"}; Nodo m = {.tipo='B', .op='*', .izq=&n2, .der=&n3}; Nodo s = {.tipo='B', .op='+', .izq=&x, .der=&m}; char l[8][64] = {{0}}; char r[16] = ""; generar_tac_opt(&s, l, 8, r, 16); verificar_str("x + 2 * 3 (linea)", "_t0 = x + 6", l[0]); }
    { Nodo x = {.tipo='V', .valor="x"}; Nodo n2 = {.tipo='N', .valor="2"}; Nodo n3 = {.tipo='N', .valor="3"}; Nodo m = {.tipo='B', .op='*', .izq=&n2, .der=&n3}; Nodo s = {.tipo='B', .op='+', .izq=&x, .der=&m}; char l[8][64] = {{0}}; char r[16] = ""; generar_tac_opt(&s, l, 8, r, 16); verificar_str("x + 2 * 3 (resultado)", "_t0", r); }
    { Nodo x = {.tipo='V', .valor="x"}; Nodo y = {.tipo='V', .valor="y"}; Nodo n2 = {.tipo='N', .valor="2"}; Nodo n3 = {.tipo='N', .valor="3"}; Nodo p = {.tipo='B', .op='*', .izq=&x, .der=&y}; Nodo m = {.tipo='B', .op='*', .izq=&n2, .der=&n3}; Nodo s = {.tipo='B', .op='+', .izq=&p, .der=&m}; char l[8][64] = {{0}}; char r[16] = ""; generar_tac_opt(&s, l, 8, r, 16); verificar_str_largo("x * y + 2 * 3", "_t1 = _t0 + 6", l[1]); }
    { Nodo n2 = {.tipo='N', .valor="2"}; Nodo n3 = {.tipo='N', .valor="3"}; Nodo n4 = {.tipo='N', .valor="4"}; Nodo n5 = {.tipo='N', .valor="5"}; Nodo s1 = {.tipo='B', .op='+', .izq=&n2, .der=&n3}; Nodo s2 = {.tipo='B', .op='+', .izq=&n4, .der=&n5}; Nodo m = {.tipo='B', .op='*', .izq=&s1, .der=&s2}; char l[8][64] = {{0}}; char r[16] = ""; int n = generar_tac_opt(&m, l, 8, r, 16); verificar_int("todo constante: sin lineas", (long)(0), (long)(n)); }
    { Nodo n2 = {.tipo='N', .valor="2"}; Nodo n3 = {.tipo='N', .valor="3"}; Nodo n4 = {.tipo='N', .valor="4"}; Nodo n5 = {.tipo='N', .valor="5"}; Nodo s1 = {.tipo='B', .op='+', .izq=&n2, .der=&n3}; Nodo s2 = {.tipo='B', .op='+', .izq=&n4, .der=&n5}; Nodo m = {.tipo='B', .op='*', .izq=&s1, .der=&s2}; char l[8][64] = {{0}}; char r[16] = ""; generar_tac_opt(&m, l, 8, r, 16); verificar_str("todo constante: resultado", "45", r); }
    { Nodo x = {.tipo='V', .valor="x"}; Nodo z = {.tipo='N', .valor="0"}; Nodo s = {.tipo='B', .op='+', .izq=&x, .der=&z}; char l[4][64] = {{0}}; char r[16] = ""; generar_tac_opt(&s, l, 4, r, 16); verificar_str("x + 0 (identidad)", "x", r); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion30.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion30_c.c artesano_sesion30_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion30_c.c ia_sesion30_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion30.md` a `auditorias/auditoria_sesion30.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
